# nb27 - H2: shower-template decomposition

Equations: e_i = s_i + p_i with s_i = E*f(r_i;theta)+delta_i and p_i >= 0. The clean sample is a free library of pure-signal showers, so the template f can be LEARNED from clean per-cell fractions (target f_i = e_i/sum(e) - observable, no truth labels needed). On min-bias each cell then votes for the photon energy, v_i = e_i / f_hat_i; pileup only ever inflates votes (p_i >= 0), so a **weighted low quantile of the votes** is robust to contamination. A timing variant drops votes from measured out-of-time cells.

Targets from nb26: clean per-bin floor 0.028-0.038 (E>11 GeV); min-bias GateHuber ensemble per-bin 0.038-0.049; the gap (0.018-0.031 per bin) is the pileup term this estimator attacks. Success = min-bias per-bin curve approaching the clean floor, i.e. ~0.030-0.035 at mid-high E.

In [1]:
import os, sys, pathlib, copy, time
import numpy as np, pandas as pd, uproot, awkward as ak, matplotlib.pyplot as plt
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS, CELL_KEYS
DATASETS = {
    'clean':   sorted((REPO / 'data' / 'full').glob('matched_*.root')),
    'minbias': sorted((REPO / 'data' / 'minimum_bias').glob('matched_*.root')),
}
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
FIG = REPO / 'reports' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODE = os.environ.get('NB27_MODE', 'full')
print('device', DEVICE, '| mode', MODE)

device cuda | mode full


## Build (all cells, cap 240): energies + geometry + seed-relative time per cell

In [2]:
TKEYS = CELL_KEYS + ['cell_times_front']
AUX = ['sig_flux_prod_vertex_z', 'sig_flux_eTot']
LCAP = 240
def build_all(files, vertex_max=100.0):
    O = {k: [] for k in ['e', 'geo', 'dt', 'hasv', 'glob', 'Etrue']}
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TKEYS + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        for i in np.flatnonzero(vz < vertex_max):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TKEYS}
            e = cc['energy']
            if len(e) == 0: continue
            seed = int(np.argmax(e))
            x = cc['cell_x']; yy = cc['cell_y']; ix = cc['imodx']; iy = cc['jmody']
            pts = np.stack([x, yy], 1)
            pitch = np.full(len(x), np.nan)
            for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
                sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
                if len(p) >= 2:
                    d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
                    pitch[sel] = np.median(np.min(d, axis=1))
            fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
            pitch[~np.isfinite(pitch)] = fill
            mod = np.array([int(np.argmin(np.abs(PITCH - p))) for p in pitch], dtype=int)
            rx = x - x[seed]; ry = yy - yy[seed]; rdr = np.hypot(rx, ry)
            o = np.argsort(rdr)[:LCAP]
            e2 = e[o]; rx, ry, rdr, pit, md_ = rx[o], ry[o], rdr[o], pitch[o], mod[o]
            tf = cc['cell_times_front'][o]
            vf = np.abs(tf) < 1e6
            t0 = tf[0] if vf[0] else (np.median(tf[vf]) if vf.any() else 0.0)
            dt = np.where(vf, tf - t0, 0.0)
            sumE = float(e2.sum())
            cogx = float((e2 * rx).sum() / (sumE + EPS)); cogy = float((e2 * ry).sum() / (sumE + EPS))
            oh = np.zeros((len(e2), len(PITCH))); oh[np.arange(len(e2)), md_] = 1.0
            geo = np.concatenate([np.stack([rx/pit, ry/pit, rdr/pit, np.log(pit)], 1), oh], 1)
            O['e'].append(e2.astype(np.float32)); O['geo'].append(geo.astype(np.float32))
            O['dt'].append(dt.astype(np.float32)); O['hasv'].append(vf.astype(np.float32))
            O['glob'].append([np.log1p(sumE), np.log(len(e2)+1.0), cogx/pit[0], cogy/pit[0]])
            O['Etrue'].append(float(a['sig_flux_eTot'][i]))
    O['glob'] = np.array(O['glob'], np.float32); O['Etrue'] = np.array(O['Etrue'])
    print(f'{len(O["Etrue"])} clusters')
    return O
NF = None if MODE == 'full' else 8
CL = build_all(DATASETS['clean'][:NF] if NF else DATASETS['clean'])
MB = build_all(DATASETS['minbias'][:NF] if NF else DATASETS['minbias'])

36852 clusters
89797 clusters


## TemplateNet: per-cell logits -> softmax = predicted photon energy fraction
Trained on clean where the target fraction e_i/sum(e) is fully observable (all deposits are signal). Loss = KL(target || softmax). O(N) per cell - light GPU load.

In [3]:
class TemplateNet(nn.Module):
    def __init__(self, gdim=9, cdim=4, d=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(gdim + cdim, d), nn.GELU(), nn.Linear(d, d), nn.GELU(),
                                 nn.Linear(d, 1))
    def forward(self, geo, glob, m):
        B, L, _ = geo.shape
        z = torch.cat([geo, glob.unsqueeze(1).expand(B, L, -1)], -1)
        logit = self.net(z).squeeze(-1)
        logit = logit.masked_fill(~m, -1e9)
        return torch.log_softmax(logit, -1)
def pad_ds(O):
    N = len(O['Etrue']); maxL = max(len(e) for e in O['e'])
    E = np.zeros((N, maxL), np.float32); GEO = np.zeros((N, maxL, 9), np.float32)
    DT = np.zeros((N, maxL), np.float32); HV = np.zeros((N, maxL), np.float32)
    M = np.zeros((N, maxL), np.bool_)
    for i in range(N):
        L = len(O['e'][i]); E[i, :L] = O['e'][i]; GEO[i, :L] = O['geo'][i]
        DT[i, :L] = O['dt'][i]; HV[i, :L] = O['hasv'][i]; M[i, :L] = True
    return dict(E=E, GEO=GEO, DT=DT, HV=HV, M=M, glob=O['glob'], Etrue=O['Etrue'], maxL=maxL)
cl = pad_ds(CL); mb = pad_ds(MB)
ncl = len(cl['Etrue']); ctr, cva, cte = (np.flatnonzero((cl['Etrue']>=1)&(cl['Etrue']<=100))[s]
                                         for s in split(int(((cl['Etrue']>=1)&(cl['Etrue']<=100)).sum())))
nmb = len(mb['Etrue']); mkeep = np.flatnonzero((mb['Etrue']>=1)&(mb['Etrue']<=100))
mtr, mva, mte = (mkeep[s] for s in split(len(mkeep)))
print('clean', ncl, 'minbias', nmb)

clean 36852 minbias 89797


In [4]:
EPOCHS = {'smoke': 12, 'full': 40}[MODE]
torch.manual_seed(0)
tpl = TemplateNet().to(DEVICE)
opt = torch.optim.AdamW(tpl.parameters(), lr=1e-3, weight_decay=1e-4)
Ec = torch.from_numpy(cl['E']).to(DEVICE); Gc = torch.from_numpy(cl['GEO']).to(DEVICE)
Mc = torch.from_numpy(cl['M']).to(DEVICE); GLc = torch.from_numpy(cl['glob']).to(DEVICE)
frac_t = Ec / Ec.sum(1, keepdim=True).clamp(min=1e-9)
rng = np.random.default_rng(0)
for ep in range(EPOCHS):
    tpl.train()
    idx = rng.permutation(ctr)
    tot = 0.0; nb_ = 0
    for j in range(0, len(idx), 512):
        b = torch.from_numpy(idx[j:j+512]).to(DEVICE)
        logf = tpl(Gc[b], GLc[b], Mc[b])
        loss = -(frac_t[b] * logf).sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); nb_ += 1
    if ep % 10 == 0 or ep == EPOCHS-1: print(f'epoch {ep} loss {tot/max(nb_,1):.4f}', flush=True)

epoch 0 loss 2.9008
epoch 10 loss 1.2794
epoch 20 loss 1.2692
epoch 30 loss 1.2660
epoch 39 loss 1.2645


## Estimators
Per-cell votes are too noisy (shower fluctuations), so the workhorse is a **joint fit**: matched filter E_LS = sum(f_i e_i)/sum(f_i^2), made pileup-robust by iterative ONE-SIDED reweighting - cells whose residual e_i - E*f_i is strongly positive (pileup can only add energy) get down-weighted, then refit. `tau` variant additionally zeroes the weight of measured out-of-time cells. Log-linear calibration on the train split.

In [5]:
def template_fracs(P):
    E = torch.from_numpy(P['E']).to(DEVICE); G = torch.from_numpy(P['GEO']).to(DEVICE)
    M = torch.from_numpy(P['M']).to(DEVICE); GL = torch.from_numpy(P['glob']).to(DEVICE)
    fr = np.zeros(P['E'].shape, np.float32)
    tpl.eval()
    with torch.no_grad():
        for j in range(0, len(fr), 1024):
            b = torch.arange(j, min(j+1024, len(fr)), device=DEVICE)
            fr[j:j+1024] = torch.exp(tpl(G[b], GL[b], M[b])).cpu().numpy()
    return fr
def robust_fit(P, fr, k=2.0, iters=4, tau=None):
    E = P['E']; M = P['M'].astype(np.float32)
    if tau is not None:
        M = M * (((P['HV'] < 0.5) | (np.abs(P['DT']) <= tau)).astype(np.float32))
    f = fr * M
    u = np.ones_like(f)
    Eh = (f * E).sum(1) / np.maximum((f * f).sum(1), 1e-9)
    for _ in range(iters):
        r = E - Eh[:, None] * fr
        scale = np.median(np.abs(r) + 1e-9, axis=1, keepdims=True) + 1e-9
        u = np.where(r > k * scale, (k * scale) / np.maximum(r, 1e-9), 1.0) * M
        Eh = (u * f * E).sum(1) / np.maximum((u * f * fr).sum(1), 1e-9)
    return np.maximum(Eh, 1e-3)
def calib_eval(raw, P, tr, te):
    lv = np.log(np.maximum(raw, EPS))
    a, b = np.polyfit(lv[tr], np.log(P['Etrue'][tr]), 1)
    pe = np.exp(a * lv[te] + b)
    return resolution(pe, P['Etrue'][te])['sigma_eff'], pe

## Sanity on clean, then min-bias (quantile scan on val, report on test)

In [6]:
fr_cl = template_fracs(cl)
cov = np.corrcoef((fr_cl[cl['M']]).ravel(), (cl['E']/cl['E'].sum(1, keepdims=True).clip(1e-9))[cl['M']].ravel())[0,1]
print('template-vs-actual fraction corr (clean):', round(float(cov), 3))
raw = robust_fit(cl, fr_cl, k=1e9, iters=0)
s, _ = calib_eval(raw, cl, ctr, cte)
print(f'clean matched-filter (no reweight): sigma_eff {s:.4f}   (all-cell sum anchor 0.0815; per-bin floor 0.028-0.038)')
for k in [1.5, 2.5]:
    raw = robust_fit(cl, fr_cl, k=k)
    s, _ = calib_eval(raw, cl, ctr, cte)
    print(f'clean robust k={k}: sigma_eff {s:.4f}')

template-vs-actual fraction corr (clean): 0.996
clean matched-filter (no reweight): sigma_eff 0.0919   (all-cell sum anchor 0.0815; per-bin floor 0.028-0.038)
clean robust k=1.5: sigma_eff 0.4938
clean robust k=2.5: sigma_eff 0.4901


In [7]:
fr_mb = template_fracs(mb)
best = None
for k in [1.0, 1.5, 2.0, 3.0, 1e9]:
    for tau in [None, 1.5]:
        raw = robust_fit(mb, fr_mb, k=k, iters=0 if k > 1e8 else 4, tau=tau)
        sv, _ = calib_eval(raw, mb, mtr, mva)
        if best is None or sv < best[0]: best = (sv, k, tau)
        print(f'val k={k} tau={tau}: {sv:.4f}', flush=True)
sv, k, tau = best
raw = robust_fit(mb, fr_mb, k=k, iters=0 if k > 1e8 else 4, tau=tau)
st, pe_t = calib_eval(raw, mb, mtr, mte)
print(f'BEST on val: k={k} tau={tau} -> TEST sigma_eff {st:.4f}  (GateHuber kNN-25 ens anchor 0.0463)')

val k=1.0 tau=None: 0.5723
val k=1.0 tau=1.5: 0.5725
val k=1.5 tau=None: 0.5711
val k=1.5 tau=1.5: 0.5718
val k=2.0 tau=None: 0.5695
val k=2.0 tau=1.5: 0.5702
val k=3.0 tau=None: 0.5657
val k=3.0 tau=1.5: 0.5679
val k=1000000000.0 tau=None: 0.4543
val k=1000000000.0 tau=1.5: 0.4548
BEST on val: k=1000000000.0 tau=None -> TEST sigma_eff 0.4547  (GateHuber kNN-25 ens anchor 0.0463)


## Per-bin verdict vs the nb26 gap table

In [8]:
te_e = mb['Etrue'][mte]
edges = np.array([1.3, 11.0, 17.6, 24.4, 35.1, 54.0, 100.0])
gh = pd.read_csv(OUT / 'minbias__GateHuber.csv')
blocks = [gh[gh.seed == s].reset_index(drop=True) for s in sorted(gh['seed'].unique())]
n2 = min(len(b) for b in blocks)
gh_true = blocks[0]['true_energy'].to_numpy()[:n2]
gh_ens = np.stack([b['pred_energy'].to_numpy()[:n2] for b in blocks]).mean(0)
CLEAN_FLOOR = {0: 0.0567, 1: 0.0382, 2: 0.0332, 3: 0.0319, 4: 0.0282, 5: 0.0314}
print(f'{"E bin [GeV]":>16s} {"template":>9s} {"GateHuber":>10s} {"clean floor":>11s}')
rows = []
for i in range(6):
    hi = edges[i+1] + (1e-9 if i == 5 else 0)
    mm = (te_e >= edges[i]) & (te_e < hi); mg = (gh_true >= edges[i]) & (gh_true < hi)
    if mm.sum() < 20: continue
    s1 = resolution(pe_t[mm], te_e[mm])['sigma_eff']
    s2 = resolution(gh_ens[mg], gh_true[mg])['sigma_eff'] if mg.sum() >= 20 else float('nan')
    rows.append(dict(bin=f'{edges[i]:.1f}-{edges[i+1]:.1f}', template=s1, gatehuber=s2, floor=CLEAN_FLOOR[i]))
    print(f'{edges[i]:7.1f}-{edges[i+1]:6.1f} {s1:9.4f} {s2:10.4f} {CLEAN_FLOOR[i]:11.4f}')
pd.DataFrame(rows).to_csv(OUT / 'nb27_template.csv', index=False)
print('saved nb27_template.csv')

     E bin [GeV]  template  GateHuber clean floor
    1.3-  11.0    0.4888     0.0763      0.0567
   11.0-  17.6    0.2591     0.0490      0.0382
   17.6-  24.4    0.1890     0.0377      0.0332
   24.4-  35.1    0.2127     0.0377      0.0319
   35.1-  54.0    0.2591     0.0391      0.0282
   54.0- 100.0    0.2589     0.0418      0.0314
saved nb27_template.csv
